In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LogAnalytics") \
    .master("local[*]") \
    .getOrCreate()

print("Spark initialized")

Spark initialized


In [ ]:
from pyspark.sql.functions import regexp_extract, col, to_timestamp

log_path = "/content/drive/MyDrive/nasa_logs.txt"

df = spark.read.text(log_path)

log_pattern = r'(\S+) - - \[(.*?)\] "(\S+) (\S+) .*?" (\d{3}) (\S+)'

parsed_df = df.select(
    regexp_extract('value', log_pattern, 1).alias('ip'),
    regexp_extract('value', log_pattern, 2).alias('timestamp'),
    regexp_extract('value', log_pattern, 3).alias('method'),
    regexp_extract('value', log_pattern, 4).alias('endpoint'),
    regexp_extract('value', log_pattern, 5).cast('int').alias('status'),
    regexp_extract('value', log_pattern, 6).alias('bytes')
)

parsed_df = parsed_df.withColumn("bytes", col("bytes").cast("int"))

parsed_df = parsed_df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"), "dd/MMM/yyyy:HH:mm:ss Z")
)

parsed_df = parsed_df.dropna()

parsed_df.show(5)

+--------------------+-------------------+------+--------------------+------+-----+
|                  ip|          timestamp|method|            endpoint|status|bytes|
+--------------------+-------------------+------+--------------------+------+-----+
|        199.72.81.55|1995-07-01 04:00:01|   GET|    /history/apollo/|   200| 6245|
|unicomp6.unicomp.net|1995-07-01 04:00:06|   GET| /shuttle/countdown/|   200| 3985|
|      199.120.110.21|1995-07-01 04:00:09|   GET|/shuttle/missions...|   200| 4085|
|  burger.letters.com|1995-07-01 04:00:11|   GET|/shuttle/countdow...|   304|    0|
|      199.120.110.21|1995-07-01 04:00:11|   GET|/shuttle/missions...|   200| 4179|
+--------------------+-------------------+------+--------------------+------+-----+
only showing top 5 rows



In [ ]:
from pyspark.sql.functions import date_format

# Convert timestamp to string before toPandas to avoid casting issues in Python 3.12/Pandas 2.0
export_df = parsed_df.limit(10000).withColumn("timestamp", date_format("timestamp", "yyyy-MM-dd HH:mm:ss"))

sample_df = export_df.toPandas()

# Save to CSV
sample_df.to_csv("logs_sample.csv", index=False)

print("Saved logs_sample.csv")
display(sample_df.head())

Saved logs_sample.csv


,ip,timestamp,method,endpoint,status,bytes
0,199.72.81.55,1995-07-01 04:00:01,GET,/history/apollo/,200,6245
1,unicomp6.unicomp.net,1995-07-01 04:00:06,GET,/shuttle/countdown/,200,3985
2,199.120.110.21,1995-07-01 04:00:09,GET,/shuttle/missions/sts-73/mission-sts-73.html,200,4085
3,burger.letters.com,1995-07-01 04:00:11,GET,/shuttle/countdown/liftoff.html,304,0
4,199.120.110.21,1995-07-01 04:00:11,GET,/shuttle/missions/sts-73/sts-73-patch-small.gif,200,4179


In [ ]:
from pyspark.sql.functions import count, window

# Requests over time
requests_over_time = parsed_df.groupBy(
    window(col("timestamp"), "1 hour")
).count().orderBy("window")

# Top endpoints
top_endpoints = parsed_df.groupBy("endpoint") \
    .count() \
    .orderBy(col("count").desc()) \
    .limit(10)

# Status distribution
status_counts = parsed_df.groupBy("status").count()

# Error rate
total = parsed_df.count()
errors = parsed_df.filter(col("status") >= 400).count()
error_rate = errors / total if total > 0 else 0

print("Error Rate:", error_rate)

Error Rate: 3.3705178131866425e-05


In [ ]:
requests_over_time.toPandas().to_json("requests_over_time.json", orient="records")
top_endpoints.toPandas().to_json("top_endpoints.json", orient="records")
status_counts.toPandas().to_json("status_counts.json", orient="records")

import json
with open("aggregated_metrics.json", "w") as f:
    json.dump({"error_rate": error_rate}, f)

print("Saved all metrics")

Saved all metrics


In [ ]:
from pyspark.sql.functions import mean, stddev

stats = requests_over_time.select(
    mean("count").alias("mean"),
    stddev("count").alias("std")
).collect()[0]

mean_val = stats["mean"]
std_val = stats["std"]

threshold = mean_val + 2 * std_val

anomalies = requests_over_time.filter(col("count") > threshold)

anomalies.toPandas().to_json("anomalies.json", orient="records")

print("Anomalies detected and saved")

Anomalies detected and saved


In [ ]:
from google.colab import files

files.download("logs_sample.csv")
files.download("requests_over_time.json")
files.download("top_endpoints.json")
files.download("status_counts.json")
files.download("aggregated_metrics.json")
files.download("anomalies.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>